In [1]:
from dataclasses import dataclass
import os
from pathlib import Path

## Entity

In [2]:
@dataclass(frozen=True)
class PrepareBaseModelConfig:
    root_dir: Path
    base_model_path: Path
    updated_base_model_path: Path
    params_image_size: list
    params_learning_rate: float
    params_include_top: bool
    params_weights: str
    params_classes: int

In [3]:
os.getcwd()

'd:\\AI-Food-Recognition-Nutrition-Assistant\\research'

In [4]:
os.chdir("..")

In [5]:
%pwd

'd:\\AI-Food-Recognition-Nutrition-Assistant'

## Configuration Manager

In [6]:
from AI_Food_Recognition_Nutrition_Assistant.constants import *
from AI_Food_Recognition_Nutrition_Assistant.utils.common import read_yaml,create_directories,decodeImage

In [7]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH
                 ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_prepare_base_model_config(self) -> PrepareBaseModelConfig:
        config = self.config.prepare_base_model

        create_directories([config.root_dir])
        prepare_base_model_config = PrepareBaseModelConfig(
            root_dir = Path(config.root_dir),
            base_model_path = Path(config.base_model_path),
            updated_base_model_path = Path(config.updated_base_model_path),
            params_image_size = self.params.IMAGE_SIZE,
            params_learning_rate = self.params.LEARNING_RATE ,
            params_include_top = self.params.INCLUDE_TOP,
            params_weights = self.params.WEIGHTS,
            params_classes = self.params.CLASSES,  
        )

        return prepare_base_model_config    

## Prepare base model Component

In [ ]:
from AI_Food_Recognition_Nutrition_Assistant import logger
from AI_Food_Recognition_Nutrition_Assistant.utils.common import get_size
import tensorflow

In [25]:
class PrepareBaseModel:
    
    def __init__(self, config: PrepareBaseModelConfig):
        self.config = config


    def get_base_model(self):
        self.model = tensorflow.keras.applications.VGG16(
            include_top=self.config.params_include_top,
            weights=self.config.params_weights,
            input_shape=self.config.params_image_size,
        )
        self.save_model(path = self.config.base_model_path, model = self.model) 

    @staticmethod
    def _prepare_full_model(model,classes,freeze_all,freeze_till,learning_rate):
        if freeze_all:
            for layer in model.layers:
                layer.trainable = False
        elif (freeze_till is not None) and (freeze_till > 0):
            for layer in model.layers[:-freeze_till]:
                layer.trainable = False

        flatten_in = tensorflow.keras.layers.Flatten()(model.output)
        predication = tensorflow.keras.layers.Dense(
            units=classes,
            activation='softmax'
        )(flatten_in)
        
        full_model = tensorflow.keras.models.Model(
            inputs=model.input,
            outputs=predication
        )

        full_model.compile(
            optimizer=tensorflow.keras.optimizers.SGD(learning_rate = learning_rate),
            loss=tensorflow.keras.losses.CategoricalCrossentropy(),
            metrics=["accuracy"],
        )

        full_model.summary()
        return full_model
    

    def update_base_model(self):
        self.full_model = self._prepare_full_model(
            model=self.model,
            classes=self.config.params_classes,
            freeze_all=True,
            freeze_till=None,
            learning_rate=self.config.params_learning_rate
        )

        self.save_model(path = self.config.updated_base_model_path, model = self.full_model)

    
    @staticmethod
    def save_model(path: Path, model: tensorflow.keras.Model):
        model.save(path)

In [26]:
try:
    config = ConfigurationManager()
    prepare_base_model_config = config.get_prepare_base_model_config()
    prepare_base_model = PrepareBaseModel(config=prepare_base_model_config)
    prepare_base_model.get_base_model()
    prepare_base_model.update_base_model()
except Exception as e:
    raise e

[2026-03-16 18:44:05,629: INFO: common: yaml file: <_io.TextIOWrapper name='config\\config.yaml' mode='r' encoding='cp1252'> loaded successfully]
[2026-03-16 18:44:05,631: INFO: common: yaml file: <_io.TextIOWrapper name='params.yaml' mode='r' encoding='cp1252'> loaded successfully]
[2026-03-16 18:44:05,632: INFO: common: created directory at: artifacts]
[2026-03-16 18:44:05,634: INFO: common: created directory at: artifacts/prepare_base_model]
[2026-03-16 18:44:06,057: WARNING: saving_utils: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_6 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
     

d:\AI-Food-Recognition-Nutrition-Assistant\.venv\Lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


## Pipeline

In [40]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zipfile()
except Exception as e:
    raise e

[2026-03-13 16:59:07,505: INFO: common: yaml file: <_io.TextIOWrapper name='config\\config.yaml' mode='r' encoding='cp1252'> loaded successfully]
[2026-03-13 16:59:07,506: INFO: common: yaml file: <_io.TextIOWrapper name='params.yaml' mode='r' encoding='cp1252'> loaded successfully]
[2026-03-13 16:59:07,508: INFO: common: created directory at: artifacts]
[2026-03-13 16:59:07,509: INFO: common: created directory at: artifacts/data_ingestion]
[2026-03-13 17:03:47,050: INFO: 3471712837: artifacts/data_ingestion/data.zip downloaded successfully with info Content-Type: application/zip
X-GUploader-UploadID: AGQBYWyn3laG6kHzV2yQfCCOoYvC11Z9QNw6xCbl5mQb__4cwia-rneDQ30bTEUvP4M0URcP
Expires: Fri, 13 Mar 2026 15:59:09 GMT
Date: Fri, 13 Mar 2026 15:59:09 GMT
Cache-Control: private, max-age=0
Last-Modified: Thu, 21 Nov 2019 01:36:33 GMT
ETag: "4c697f34e0f1b5db4b95cb3793c02592"
x-goog-generation: 1574300193295556
x-goog-metageneration: 1
x-goog-stored-content-encoding: identity
x-goog-stored-content